Installation

In [23]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

Import Required Libraries

In [24]:
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import json
from datasets import Dataset

Model Configuration & 4-bit Quantized Model Options

In [25]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

Load Pretrained Base Model

In [ ]:
model_name = "unsloth/Meta-Llama-3.1-8B"
model, tokenizer = FastLanguageModel.from_pretrained(model_name= "unsloth/Meta-Llama-3.1-8B", max_seq_length = max_seq_length, dtype = None, load_in_4bit = True)

Inference Before Fine-Tuning

In [ ]:
question = "What is Soda Thakkai Aatam" # Soda Thakkai Aatam is a traditional Tamil village game
                                        # played using soda bottle caps (thakkai).
inputs = tokenizer(question, return_tensors= 'pt').to("cuda")
outputs = model.generate(**inputs, max_new_tokens= 100)
print(f"Before Fine-Tuning :\n{tokenizer.decode(outputs[0],skip_special_tokens = True)}")

Load and Prepare Training Dataset

In [ ]:
with open("Soda_thakkai_game_data.json","r")as f:
  data = json.load(f)
dataset = Dataset.from_list(data)
def format_data(example):
  return {"text": f"### Instruction : {example['instruction']}\n### Response: {example['output']}"}
dataset = dataset.map(format_data)
print(dataset)
print("\n")
print(data)

Apply LoRA (PEFT) to the Base Model

In [ ]:
model = FastLanguageModel.get_peft_model(model,
    r=16, # Choose any number > 0 (8, 16, 32, 64, 128) --- number of LoRA Layers
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    bias="none",
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Configure Supervised Fine-Tuning (SFTTrainer)

In [ ]:
trainer = SFTTrainer(model= model, tokenizer= tokenizer, train_dataset= dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(output_dir= "./ Soda_thakkai_game_finetuned",
        # Batch settings
           per_device_train_batch_size = 2,
           gradient_accumulation_steps = 4,
        # Training control (CHOOSE ONE: epochs OR max_steps)
          #  num_train_epochs = 3,
           max_steps = 60,
        # Learning
           learning_rate = 0.0002,
           lr_scheduler_type = "linear",
           warmup_steps = 5,
        # Optimization
           optim = "adamw_8bit",
           weight_decay = 0.01,
        # Precision
           fp16=not torch.cuda.is_bf16_supported(),
           bf16 = torch.cuda.is_bf16_supported(),
        # Logging & saving
           save_strategy = "epoch",
           logging_steps = 1,
        # Misc
           report_to = "none",
           seed = 3407,
           )
    )

Start Model Training

In [ ]:
trainer_stats = trainer.train()

Save Merged Fine-Tuned Model

In [ ]:
model.save_pretrained_merged("Soda_thakka_model", tokenizer)

Load the Fine-Tuned Merged Model for Inference

In [ ]:
from unsloth import FastLanguageModel
import torch

modelft, tokenizerft = FastLanguageModel.from_pretrained(model_name= "Soda_thakka_model", load_in_4bit = True)

Interactive Chat Loop (Fine-Tuned Model)

In [ ]:
chat_history = ""
while True:
    question = input("Enter your question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Chat ended.. Tata")
        break
    chat_history += f"User: {question}\nAssistant: "
    inputs = tokenizerft(chat_history, return_tensors='pt').to("cuda")
    outputs = modelft.generate(**inputs, max_new_tokens=100)

    response = tokenizerft.decode(outputs[0], skip_special_tokens=True)

    # Extract only latest assistant response
    response = response.split("Assistant:")[-1].strip()

    chat_history += response + "\n"

    print("\nAfter Fine-Tuning:\n", response)
    print("-" * 50)